System Path Setup

In [36]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [37]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import pandas as pd
import geopandas as gpd
import folium # For interactive mapping
from pygbif import occurrences # For GBIF data
from configs.regions import kenyan_coast_roi # Your ROI

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID

print("All core libraries imported and GEE initialized.")

All core libraries imported and GEE initialized.


Load Cleaned GBIF Data and Mangrove Extent

In [38]:
# Cell 3: Load Cleaned GBIF Data and GEE Mangrove Extent (REVISED FOR ULTIMATE CLEANLINESS)
print("--- Loading Cleaned GBIF Data and Mangrove Extent ---")

# Load the cleaned GBIF GeoJSON locally
gbif_cleaned_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_coastal_occurrences_cleaned.geojson')
if os.path.exists(gbif_cleaned_path):
    gbif_gdf_local = gpd.read_file(gbif_cleaned_path)
    print(f"Loaded {len(gbif_gdf_local)} cleaned GBIF records locally.")
else:
    print(f"Error: Cleaned GBIF data not found at {gbif_cleaned_path}. Please re-run 04_Species_GBIF_Acquisition.ipynb.")
    sys.exit("No cleaned GBIF data to process.")

# --- REVISED: Create GEE FeatureCollection with ONLY essential, clean properties ---
# This prevents any problematic properties from the original GBIF data (like 'http://unknown.org/nick')
# from being carried over into the GEE environment.

# List the essential columns you need from the local GeoDataFrame
essential_local_cols = [
    'gbifID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus',
    'species', 'eventDate', 'decimalLatitude', 'decimalLongitude'
]

# Filter the local GeoDataFrame to only these essential columns and geometry
# Also, ensure no NaNs in lat/lon before converting
gbif_gdf_clean_for_gee = gbif_gdf_local[essential_local_cols + ['geometry']].dropna(subset=['decimalLatitude', 'decimalLongitude'])

# Convert to a list of ee.Feature objects
# This is a client-side operation to prepare data for GEE.
features_list = []
for index, row in gbif_gdf_clean_for_gee.iterrows():
    # Create a dictionary of properties for this feature
    properties = {col: row[col] for col in essential_local_cols}
    
    # Handle potential non-serializable types if any (e.g., convert objects to string)
    for key, value in properties.items():
        if pd.api.types.is_object_dtype(type(value)):
            properties[key] = str(value)
    
    # Create an ee.Geometry from the shapely geometry
    ee_geometry = ee.Geometry.Point([row.geometry.x, row.geometry.y])
    
    features_list.append(ee.Feature(ee_geometry, properties))

# Create the GEE FeatureCollection
gbif_fc_gee = ee.FeatureCollection(features_list)
print(f"Converted {gbif_fc_gee.size().getInfo()} (cleaned) GBIF records to GEE FeatureCollection with essential properties only.")


# Load the GEE Mangrove Extent image (from 01_Mangrove_Distribution_GMW.ipynb)
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")
mangroves_raster_mask = recent_mangrove_image.clip(kenyan_coast_roi).gt(0).unmask(0).rename('is_mangrove')
print("GEE Mangrove Extent raster mask loaded and prepared for sampling.")

--- Loading Cleaned GBIF Data and Mangrove Extent ---
Loaded 209 cleaned GBIF records locally.
Converted 209 (cleaned) GBIF records to GEE FeatureCollection with essential properties only.
GEE Mangrove Extent raster mask loaded and prepared for sampling.


 Spatially Filter GBIF Points to Mangrove Areas

In [ ]:
# Cell 4: Spatially Filter GBIF Points by Sampling Mangrove Raster (FULL REVISED METHOD WITH BUFFER)
print("--- Spatially Filtering GBIF data to Mangrove Extent (Alternative Method with Buffer) ---")

# --- Re-load cleaned GBIF data and convert to GEE FeatureCollection ---
# This block is duplicated from Cell 3 to make Cell 4 runnable independently after kernel restart
# In a production script, you'd pass gbif_fc_gee from Cell 3 or save as GEE Asset.
gbif_cleaned_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_coastal_occurrences_cleaned.geojson')
if os.path.exists(gbif_cleaned_path):
    gbif_gdf_local = gpd.read_file(gbif_cleaned_path)
    print(f"Loaded {len(gbif_gdf_local)} cleaned GBIF records locally.")
else:
    print(f"Error: Cleaned GBIF data not found at {gbif_cleaned_path}. Please re-run 04_Species_GBIF_Acquisition.ipynb.")
    sys.exit("No cleaned GBIF data to process.")

essential_local_cols = [
    'gbifID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus',
    'species', 'eventDate', 'decimalLatitude', 'decimalLongitude'
]
gbif_gdf_clean_for_gee = gbif_gdf_local[essential_local_cols + ['geometry']].dropna(subset=['decimalLatitude', 'decimalLongitude'])

features_list = []
for index, row in gbif_gdf_clean_for_gee.iterrows():
    properties = {col: str(row[col]) if pd.api.types.is_object_dtype(type(row[col])) else row[col] for col in essential_local_cols}
    ee_geometry = ee.Geometry.Point([row.geometry.x, row.geometry.y])
    features_list.append(ee.Feature(ee_geometry, properties))
gbif_fc_gee = ee.FeatureCollection(features_list)
print(f"Converted {gbif_fc_gee.size().getInfo()} (cleaned) GBIF records to GEE FeatureCollection with essential properties only.")

# --- Load GEE Mangrove Extent image ---
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")

# Ensure the mangrove image is binary (1 for mangrove, 0 for non-mangrove) and unmask 0s explicitly.
# We rename the band to 'is_mangrove' for clarity in the sampled properties.
mangroves_raster_mask = recent_mangrove_image.clip(kenyan_coast_roi).gt(0).unmask(0).rename('is_mangrove')

# --- ADDED: Optional buffer around the mangrove mask ---
# This expands the mangrove area slightly to capture points that might be on the edge
# or have slight spatial inaccuracies, but are still relevant to the mangrove ecosystem.
# Use a small buffer, e.g., 60 meters (2 pixels at 30m resolution).
# The focal_max operation will expand the '1' values.
# We reproject to ensure the scale is consistent after focal_max.
buffered_mangroves_raster_mask = mangroves_raster_mask.focal_max(2).reproject(crs=mangroves_raster_mask.projection().crs(), scale=30)

# We update the mask again to ensure it remains a binary 0/1 mask
buffered_mangroves_raster_mask = buffered_mangroves_raster_mask.updateMask(buffered_mangroves_raster_mask.gt(0)).rename('is_mangrove')

print("GEE Mangrove Extent raster mask loaded, buffered (60m), and prepared for sampling.")

# --- Sample the 'is_mangrove' band from the *buffered* mask ---
# This adds a new property to each GBIF feature with the value of the 'is_mangrove' band at that point.
sampled_gbif_fc = buffered_mangroves_raster_mask.reduceRegions(
    collection=gbif_fc_gee,
    reducer=ee.Reducer.first(), # Get the value of the 'is_mangrove' band at each point's location
    scale=30, # Match the resolution of the mangrove image
    crs=buffered_mangroves_raster_mask.projection().crs()
)

# Filter the sampled points: keep only those where 'is_mangrove' (the output of reducer.first()) is 1.
gbif_in_mangroves_gee = sampled_gbif_fc.filter(ee.Filter.eq('first', 1))

# Rename 'first' to 'is_mangrove' (server-side)
def rename_first_to_is_mangrove(feature):
    # Use .set() to add 'is_mangrove' and .set('first', None) to effectively remove 'first'
    return feature.set('is_mangrove', feature.get('first')).set('first', None)

gbif_final_selection_gee = gbif_in_mangroves_gee.map(rename_first_to_is_mangrove)

# --- Initiate GEE Export Task ---
final_export_properties = essential_local_cols + ['is_mangrove']

# Add a check for the size BEFORE initiating export
final_collection_size = gbif_final_selection_gee.size().getInfo()
print(f"Number of GBIF records in final FeatureCollection before export: {final_collection_size}")

if final_collection_size == 0:
    print("WARNING: The final FeatureCollection is empty after filtering. Export task will be skipped.")
    print("Consider adjusting ROI or buffer size if this is unexpected.")
    gbif_in_mangroves_local = gpd.GeoDataFrame() # Assign empty DF to prevent subsequent errors
else:
    my_gee_project_id = 'gaias-ark'
    output_asset_id_prefix = f'projects/{my_gee_project_id}/assets/gaias_ark_gbif_mangrove_occurrences'
    output_description = 'GBIF Mangrove Occurrences Kenya'

    task = ee.batch.Export.table.toAsset(
        collection=gbif_final_selection_gee.select(final_export_properties),
        description=output_description,
        assetId=output_asset_id_prefix,
        selectors=final_export_properties
    )

    task.start()
    print(f"\nGEE Export Task initiated for spatially filtered GBIF data. Asset ID: {output_asset_id_prefix}")
    print("Check the 'Tasks' tab in your GEE Code Editor to monitor progress.")
    print("\nOnce the GEE task completes, you will need to load the exported asset (FeatureCollection) in a new cell:")
    print(f"   # Example to load: ee.FeatureCollection('{output_asset_id_prefix}')")

    gbif_in_mangroves_local = gpd.GeoDataFrame() # Assign empty DF for local continuation

--- Spatially Filtering GBIF data to Mangrove Extent (Alternative Method with Buffer) ---
Loaded 209 cleaned GBIF records locally.
Converted 209 (cleaned) GBIF records to GEE FeatureCollection with essential properties only.
GEE Mangrove Extent raster mask loaded, buffered (60m), and prepared for sampling.


EEException: ImageCollection.load: ImageCollection asset 'projects/GMW/Tiled/GMW_2019_v2' not found (does not exist or caller does not have access).

Visualize Spatially Filtered Data

In [23]:
# Cell 5: Visualize Spatially Filtered Data
print("--- Visualizing Spatially Filtered GBIF Data ---")

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

m_filtered = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

# Add ROI
folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.6}
).add_to(m_filtered)

# Add Mangrove Extent (for context)
mangrove_vis_params = {'min': 0, 'max': 1, 'palette': ['white', 'green']}
map_id_dict_mangroves = mangroves_extent_gee.getMapId(mangrove_vis_params)
folium.TileLayer(
    tiles=map_id_dict_mangroves['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Mangrove Extent'
).add_to(m_filtered)

# Add Spatially Filtered GBIF Points
folium.GeoJson(
    gbif_in_mangroves_local.__geo_interface__,
    name='GBIF in Mangroves',
    tooltip=folium.features.GeoJsonTooltip(fields=['scientificName'], aliases=['Species']),
    marker=folium.CircleMarker(radius=4, weight=1, color='blue', fill_color='blue', fill_opacity=0.7)
).add_to(m_filtered)

folium.LayerControl().add_to(m_filtered)
m_filtered

--- Visualizing Spatially Filtered GBIF Data ---


AttributeError: No geometry data set (expected in column 'None').